In [1]:
#import libraries
import pandas as pd
import os
import shutil
from pprint import pprint
import csv
import sqlite3
from sqlite3 import Error
from pathlib import Path
import ast
import warnings

warnings.filterwarnings("ignore")

In [2]:
#establish connection to Big Usage Database and create cursor object
conn = sqlite3.connect(
    r"C:\Users\aharrima\OneDrive - University of Tennessee\Documents\Big_Usage_Database_Updated.db"
)
cur = conn.cursor()

In [3]:
#sanity check block of code to view all the data in the Noncounter_Data table
#and make sure it looks as you expect
df = pd.read_sql_query("SELECT * from Noncounter_data;", conn)
df

,index,Report_Type,Resource,Naming_Convention,MMS_ID,Metric_Type,Annual_Statistics_Reporting,Metric_Period,Metric_Value,FY
0,0,Non-C5RT,A+T Online Library,ATArchitecture,9926518387802311,Opened Issues,None,2023-10-01 00:00:00,16.0,2024
1,1,Non-C5RT,A+T Online Library,ATArchitecture,9926518387802311,Opened Issues,None,2023-11-01 00:00:00,24.0,2024
2,2,Non-C5RT,A+T Online Library,ATArchitecture,9926518387802311,Opened Issues,None,2023-12-01 00:00:00,0.0,2024
3,3,Non-C5RT,A+T Online Library,ATArchitecture,9926518387802311,Opened Issues,None,2024-01-01 00:00:00,5.0,2024
4,4,Non-C5RT,A+T Online Library,ATArchitecture,9926518387802311,Opened Issues,None,2024-02-01 00:00:00,4.0,2024
...,...,...,...,...,...,...,...,...,...,...
9002,9002,Non-C5RT,World Shakespeare Bibliography,WorldShakespeareBibliography,9914454460102311,Uses,None,2023-02-01 00:00:00,10.0,2023
9003,9003,Non-C5RT,World Shakespeare Bibliography,WorldShakespeareBibliography,9914454460102311,Uses,None,2023-03-01 00:00:00,22.0,2023
9004,9004,Non-C5RT,World Shakespeare Bibliography,WorldShakespeareBibliography,9914454460102311,Uses,None,2023-04-01 00:00:00,7.0,2023
9005,9005,Non-C5RT,World Shakespeare Bibliography,WorldShakespeareBibliography,9914454460102311,Uses,None,2023-05-01 00:00:00,2.0,2023


In [4]:
#creating noncounter data table
#read the saved SharePoint file as a dataframe
nc = pd.read_excel(
    r"C:\Users\aharrima\OneDrive - University of Tennessee\Documents\Usage Database Update\NonCounter_Usage_Compilation.xlsx")
nc['MMS ID'] = nc['MMS ID'].astype(str)

for i in range(len(nc['MMS ID'])):
    nc['MMS ID'].iloc[i] = nc['MMS ID'].iloc[i][0:15] + '1'

#set Metric_Period type to datetime
nc['Metric_Period'] = pd.to_datetime(nc['Metric_Period'], errors = 'coerce')

#create a new FY column and populate the column with the appropriate FY for each row
nc['FY'] = nc['Metric_Period'].map(lambda x: x.year if x.month < 7 else x.year+1).astype(int).astype(int)

#display the data for sanity check
nc = nc.rename(columns = {'MMS ID': 'MMS_ID'})
nc

,Report_Type,Resource,Naming_Convention,MMS_ID,Metric_Type,Annual_Statistics_Reporting,Metric_Period,Metric_Value,FY
0,Non-C5RT,A+T Online Library,ATArchitecture,9926518387802311,Opened Issues,NaN,2023-10-01,16.0,2024
1,Non-C5RT,A+T Online Library,ATArchitecture,9926518387802311,Opened Issues,NaN,2023-11-01,24.0,2024
2,Non-C5RT,A+T Online Library,ATArchitecture,9926518387802311,Opened Issues,NaN,2023-12-01,0.0,2024
3,Non-C5RT,A+T Online Library,ATArchitecture,9926518387802311,Opened Issues,NaN,2024-01-01,5.0,2024
4,Non-C5RT,A+T Online Library,ATArchitecture,9926518387802311,Opened Issues,NaN,2024-02-01,4.0,2024
...,...,...,...,...,...,...,...,...,...
9021,Non-C5RT,World Shakespeare Bibliography,WorldShakespeareBibliography,9914454460102311,Uses,NaN,2023-02-01,10.0,2023
9022,Non-C5RT,World Shakespeare Bibliography,WorldShakespeareBibliography,9914454460102311,Uses,NaN,2023-03-01,22.0,2023
9023,Non-C5RT,World Shakespeare Bibliography,WorldShakespeareBibliography,9914454460102311,Uses,NaN,2023-04-01,7.0,2023
9024,Non-C5RT,World Shakespeare Bibliography,WorldShakespeareBibliography,9914454460102311,Uses,NaN,2023-05-01,2.0,2023


In [5]:
#if, however, you have created a new NonCounter_Usage_Compilation file for the current year's analyses,
#you can create a new table with a new name based on the year
#you will need to adjust Analysis.ipynb to account for this new table if so
nc.to_sql('Noncounter_data', conn, if_exists='replace')

9026

In [6]:
#commit the changes and close the connection
conn.commit()
conn.close()

In [ ]:
nc['MMS_ID'][1]